# Physics Law Complexity Census
**monogate v1.1.0+ — How hard are physical laws in the EML basis?**

This notebook analyses the EML approximation complexity of fundamental physical laws
vs random algebraic controls.  The key question:

> *Do physical laws occupy a privileged low-complexity region of EML tree space?*

**Short answer:** No. Physics laws are **1.7–1.8× harder** than random algebraic
expressions in the EML basis, with a single exception: the Boltzmann factor `exp(x)`,
which is the canonical 1-node EML gate.

## Three-part analysis
1. **Identity census** — prove each law as `lhs == rhs`
2. **Functional census** — measure best MSE achievable by MCTS at depth 2/4/6
3. **Rediscovery benchmark** — fit EMLRegressor to synthetic noisy data


In [ ]:
# Install if needed
# !pip install monogate matplotlib numpy -q

import sys, os
# Uncomment if running from repo root:
# sys.path.insert(0, os.path.join(os.getcwd(), 'python'))

In [ ]:
import json, os
from pathlib import Path

# Load latest census results
census_dir = Path("results/physics_census")
census_files = sorted(census_dir.glob("census_*.json")) if census_dir.exists() else []

if census_files:
    with open(census_files[-1]) as f:
        data = json.load(f)
    print(f"Loaded: {census_files[-1].name}")
    print(f"  Identity laws: {len(data['identity_census'])}")
    print(f"  Functional laws: {len(data['functional_census'])}")
    print(f"  Controls: {len(data['control_census'])}")
else:
    print("No census results found — run:")
    print("  python -m monogate.frontiers.law_complexity --n-simulations 2000 --n-controls 50 --skip-rediscovery")
    data = None

## 1. Identity Census — Proof Tier Distribution

In [ ]:
if data is None:
    print("No data — run the census first")
else:
    import matplotlib.pyplot as plt
    import numpy as np

    rows = data["identity_census"]
    proved = [r for r in rows if r["status"].startswith("proved")]
    failed = [r for r in rows if not r["status"].startswith("proved")]

    tier_names = {1: "Tier 1\nNumerical", 2: "Tier 2\nExact", 3: "Tier 3\nCertified", 4: "Tier 4\nMCTS"}
    tier_counts = {}
    for r in proved:
        t = r["tier"]
        tier_counts[t] = tier_counts.get(t, 0) + 1

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Pie: proved vs failed
    axes[0].pie(
        [len(proved), len(failed)],
        labels=[f"Proved ({len(proved)})", f"Failed ({len(failed)})"],
        colors=["#2ecc71", "#e74c3c"],
        autopct="%1.0f%%",
        startangle=90,
    )
    axes[0].set_title("Identity Laws: Prove Rate")

    # Bar: tier breakdown
    tiers = sorted(tier_counts.keys())
    counts = [tier_counts[t] for t in tiers]
    colors = ["#3498db", "#2ecc71", "#9b59b6", "#e67e22"]
    bars = axes[1].bar(
        [tier_names.get(t, f"Tier {t}") for t in tiers],
        counts,
        color=colors[:len(tiers)],
        alpha=0.85,
    )
    axes[1].set_title("Identity Laws: Proof Tier")
    axes[1].set_ylabel("Count")
    for bar, count in zip(bars, counts):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                     str(count), ha="center", fontsize=11, fontweight="bold")

    fig.suptitle("Physics Identity Census — EMLProverV2", fontsize=13)
    fig.tight_layout()
    plt.show()

    print("\nAll identities:")
    for r in rows:
        sym = "✓" if r["status"].startswith("proved") else "✗"
        print(f"  {sym} [{r['tier']}/4  {r['elapsed_s']:.3f}s] {r['name']}  ({r['category']})")

## 2. Functional Census — EML Depth vs MSE

For each law, we measure the best MSE achievable by MCTS at depth 2/4/6.

- **depth=2**: 1 EML gate (the smallest useful tree) — covers `exp(ax+b)` family  
- **depth=4**: up to 3 nested gates  
- **depth=6**: up to 5 nested gates  

**EML-native** = depth-2 MSE < 10⁻⁶ (the function is a single EML gate)

In [ ]:
if data is None:
    print("No data")
else:
    import matplotlib.pyplot as plt
    import numpy as np

    rows = data["functional_census"]

    # Clamp inf values for display
    def _mse(r, depth_key):
        v = r.get(depth_key, {}).get("mse", float("inf"))
        return min(v, 12.0)  # clamp for display

    names = [r["name"][:28] for r in rows]
    d2 = [_mse(r, "depth_2") for r in rows]
    d4 = [_mse(r, "depth_4") for r in rows]
    d6 = [_mse(r, "depth_6") for r in rows]
    native = [r.get("eml_native", False) for r in rows]
    categories = [r["category"] for r in rows]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Grouped bar by law
    x = np.arange(len(rows))
    w = 0.27
    axes[0].bar(x - w, d2, w, label="Depth 2 (1 gate)",  color="#3498db", alpha=0.85)
    axes[0].bar(x,     d4, w, label="Depth 4 (3 gates)", color="#e67e22", alpha=0.85)
    axes[0].bar(x + w, d6, w, label="Depth 6 (5 gates)", color="#2ecc71", alpha=0.85)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=55, ha="right", fontsize=7.5)
    axes[0].set_ylabel("Best MSE (clamped at 12)")
    axes[0].set_title("EML Approximation MSE by Law and Depth")
    axes[0].legend()
    axes[0].axhline(0.05, ls="--", color="gray", alpha=0.5, label="effective threshold 0.05")

    # Mark EML-native laws in red
    for i, n in enumerate(native):
        if n:
            axes[0].get_xticklabels()[i].set_color("red")
            axes[0].get_xticklabels()[i].set_fontweight("bold")

    # Category colour scatter: d2 vs d4
    cat_colors = {
        "thermodynamics": "#e74c3c",
        "statistical":    "#3498db",
        "chemistry":      "#9b59b6",
        "electromagnetism":"#f39c12",
        "quantum":        "#1abc9c",
        "mechanics":      "#2c3e50",
        "relativity":     "#e67e22",
        "information":    "#7f8c8d",
    }

    for r in rows:
        c = cat_colors.get(r["category"], "gray")
        x_val = _mse(r, "depth_2")
        y_val = _mse(r, "depth_4")
        axes[1].scatter(x_val, y_val, color=c, s=80, alpha=0.9, zorder=3)
        axes[1].annotate(r["name"][:18], (x_val, y_val), fontsize=6, alpha=0.7,
                         xytext=(3, 3), textcoords="offset points")

    # Draw y=x diagonal
    lim = max(max(d2), max(d4)) * 1.05
    axes[1].plot([0, lim], [0, lim], "k--", alpha=0.3, label="d4=d2 (no gain)")
    axes[1].set_xlabel("Depth-2 MSE")
    axes[1].set_ylabel("Depth-4 MSE")
    axes[1].set_title("Depth 2 vs Depth 4 MSE")

    # Legend for categories
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=k) for k, c in cat_colors.items()
                       if any(r["category"] == k for r in rows)]
    axes[1].legend(handles=legend_elements, fontsize=7, loc="upper left")

    fig.suptitle("Functional EML Census — Physics Laws", fontsize=13)
    fig.tight_layout()
    plt.show()

    native_count = sum(1 for r in rows if r.get("eml_native"))
    print(f"\nEML-native laws: {native_count}/{len(rows)}")
    for r in rows:
        eff = r.get("min_effective_depth")
        tag = "★ NATIVE" if r.get("eml_native") else (f"eff_d={eff}" if eff else "hard")
        d2v = r.get("depth_2", {}).get("mse", 999)
        print(f"  {tag:10s}  {r['name']:35s} d2={d2v:.3f}  [{r['category']}]")

## 3. Laws vs Random Controls — Is Physics Special?

We compare the mean MSE of physics laws vs 50 random algebraic expressions
(power laws, exp, log, trig, products).

In [ ]:
if data is None:
    print("No data")
else:
    import matplotlib.pyplot as plt
    import numpy as np

    func_rows = data["functional_census"]
    ctrl_rows = data["control_census"]

    def _collect_mse(rows, depth_key):
        vals = []
        for r in rows:
            v = r.get(depth_key, {}).get("mse", None)
            if v is not None and v < float("inf"):
                vals.append(min(v, 15.0))
        return vals

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, depth_key, label in zip(axes,
                                     ["depth_2", "depth_4", "depth_6"],
                                     ["Depth 2 (1 gate)", "Depth 4 (3 gates)", "Depth 6 (5 gates)"]):
        law_vals  = _collect_mse(func_rows, depth_key)
        ctrl_vals = _collect_mse(ctrl_rows, depth_key)

        ax.hist(ctrl_vals, bins=20, alpha=0.6, color="#3498db", label=f"Controls (n={len(ctrl_vals)})")
        ax.hist(law_vals,  bins=10, alpha=0.7, color="#e74c3c", label=f"Laws (n={len(law_vals)})")

        if law_vals:
            ax.axvline(np.mean(law_vals),  color="#c0392b", ls="--", lw=2,
                       label=f"Law mean={np.mean(law_vals):.2f}")
        if ctrl_vals:
            ax.axvline(np.mean(ctrl_vals), color="#2980b9", ls="--", lw=2,
                       label=f"Ctrl mean={np.mean(ctrl_vals):.2f}")

        ax.set_xlabel("Best MSE")
        ax.set_ylabel("Count")
        ax.set_title(label)
        ax.legend(fontsize=8)

    fig.suptitle("Physics Laws vs Random Controls — EML Approximation MSE", fontsize=13)
    fig.tight_layout()
    plt.show()

    # Comparison table
    comp = data.get("comparison", {})
    print("\nStatistical comparison:")
    print(f"{'Depth':<8} {'Law mean':>12} {'Ctrl mean':>12} {'Ratio':>8} {'Interpretation':<30}")
    for depth_key in ["depth_2", "depth_4", "depth_6"]:
        c = comp.get(depth_key, {})
        lm = c.get("law_mean")
        cm = c.get("control_mean")
        if lm and cm:
            ratio = lm / max(cm, 1e-12)
            interp = "← laws LOWER" if ratio < 0.8 else ("≈ equal" if ratio < 1.3 else "← laws HIGHER")
            print(f"{depth_key:<8} {lm:>12.3f} {cm:>12.3f} {ratio:>8.2f} {interp}")

    nr = comp.get("native_rate", {})
    print(f"\nEML-native rate: laws={nr.get('laws', 0):.0%}  controls={nr.get('controls', 0):.0%}")

## 4. The Negative-Exponent Barrier

Why can't MCTS find `exp(-x)`?

The EML grammar with leaves `{x, constants}` produces trees of the form:
```
eml(a, b) = exp(a) - ln(b)
```
To represent `exp(-x)`, we need `a = -x`. But `-x` cannot be produced by any
EML tree with non-negative constants and `x` as leaves — the grammar has no
subtraction of variables.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from monogate.search.mcts import mcts_search

x_vals = np.linspace(0.1, 3.0, 100)
target = np.exp(-x_vals)  # ground truth: exp(-x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: exp(-x) vs best EML at each depth
probe = list(np.linspace(0.1, 3.0, 80))
axes[0].plot(x_vals, target, "k-", lw=2, label="exp(-x) [ground truth]")

colors_d = ["#e74c3c", "#e67e22", "#2ecc71"]
for depth, col in zip([2, 4, 6], colors_d):
    r = mcts_search(math.exp.__class__.__instancecheck__,  # dummy — fill below
                    probe_points=None, depth=depth, n_simulations=1000, seed=42)
    # Re-run with actual function
    r = mcts_search(lambda x: math.exp(-x), probe_points=probe,
                    depth=depth, n_simulations=1000, seed=42)
    # Evaluate best formula
    from monogate.eml import eval_tree
    try:
        y_pred = np.array([eval_tree(r.best_tree, x) for x in x_vals])
        axes[0].plot(x_vals, y_pred, "--", color=col, alpha=0.8,
                     label=f"EML d={depth}: {r.best_formula[:30]}  (MSE={r.best_mse:.3f})")
    except Exception:
        axes[0].axhline(r.best_mse, color=col, ls=":", alpha=0.5,
                        label=f"d={depth} MSE={r.best_mse:.3f} (eval failed)")

axes[0].set_xlabel("x")
axes[0].set_ylabel("f(x)")
axes[0].set_title("exp(-x): best EML approximation at each depth")
axes[0].legend(fontsize=8)
axes[0].set_ylim(-0.5, 1.5)

# Right: exp(x) (EML-native) — should be perfect at depth=2
x_vals2 = np.linspace(0.0, 2.5, 100)
target2 = np.exp(x_vals2)

probe2 = list(np.linspace(0.0, 2.5, 80))
r_native = mcts_search(math.exp, probe_points=probe2, depth=2, n_simulations=500, seed=42)
axes[1].plot(x_vals2, target2, "k-", lw=2, label="exp(x) [ground truth]")

try:
    from monogate.eml import eval_tree
    y_native = np.array([eval_tree(r_native.best_tree, x) for x in x_vals2])
    axes[1].plot(x_vals2, y_native, "r--", lw=1.5,
                 label=f"EML d=2: {r_native.best_formula}  (MSE={r_native.best_mse:.1e})")
except Exception as e:
    axes[1].set_title(f"eval_tree not available: {e}")

axes[1].set_xlabel("x")
axes[1].set_title("exp(x): EML-native (perfect at depth=2)")
axes[1].legend(fontsize=9)

fig.suptitle("The Negative-Exponent Barrier in EML Grammar", fontsize=13)
fig.tight_layout()
plt.show()

## 5. Rediscovery Benchmark

Can `EMLRegressor` recover the ground-truth formula from noisy synthetic data?

In [ ]:
from monogate.frontiers.law_complexity import rediscovery_benchmark

print("Running rediscovery benchmark (8 laws)...\n")
rdisc = rediscovery_benchmark(verbose=True)

good = [r for r in rdisc if r.get("r2") is not None and r["r2"] > 0.95]
print(f"\nSuccessfully rediscovered (R² > 0.95): {len(good)}/{len(rdisc)}")
for r in good:
    print(f"  ✓ {r['name']}  R²={r['r2']:.3f}")

In [ ]:
if rdisc:
    import matplotlib.pyplot as plt
    import numpy as np

    names  = [r["name"][:28] for r in rdisc]
    r2vals = [r.get("r2") or -10 for r in rdisc]

    colors = ["#2ecc71" if v > 0.95 else ("#f39c12" if v > 0 else "#e74c3c") for v in r2vals]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(names, r2vals, color=colors, alpha=0.85)
    ax.axvline(0.95, color="green", ls="--", alpha=0.6, label="R²=0.95 threshold")
    ax.axvline(0,    color="gray",  ls=":",  alpha=0.4)
    ax.set_xlabel("R² (fit quality on noise-free data)")
    ax.set_title("EMLRegressor Rediscovery Benchmark — Physical Laws")
    ax.legend()
    ax.set_xlim(-8, 1.05)
    fig.tight_layout()
    plt.show()

## 6. Summary and Interpretation

### Key Findings

| Finding | Result |
|---------|--------|
| EML-native physics laws | 1/15 (Boltzmann weight only) |
| Identity laws proved | 14/15 (all tier-2 exact) |
| Laws harder than controls | 1.7–1.8× higher MSE |
| Rediscovery success rate | 1/8 (Boltzmann weight only) |

### The Negative-Exponent Barrier

The EML grammar `eml(a,b) = exp(a) − ln(b)` cannot express `exp(−f(x))` for
any non-trivial function f using the leaf set `{x, constants}`. This is because
producing `−f(x)` as an exponent argument requires subtraction of variables,
which is not available in the grammar.

**Consequences:**
- Radioactive decay, Arrhenius, Planck, Fermi-Dirac, Gaussian — all blocked
- Only `exp(+E/kT)` is native; `exp(−E/kT)` (the usual Boltzmann factor) is not
- Adding `−x` as a leaf would immediately unlock all these laws

### Open Questions

1. **Grammar extension**: Would adding `{x, −x}` as leaves resolve the barrier?
2. **Dual EML**: A `deml(x,y) = exp(−x) − ln(y)` gate would make all decay laws
   1-node native
3. **Are power laws also blocked?** `x^n = exp(n·ln(x))` — requires `ln(x)` as
   intermediate, which IS buildable as `−eml(0, x)` (since `eml(0,x) = 1 − ln(x)`)
